In [ ]:
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override="true")

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY=os.getenv("GOOGLE_API_KEY")

if (OPENAI_API_KEY):
    print(f"OPENAI API key exists and starts with: {OPENAI_API_KEY[:7]}")
else:
    print("OPENAI API key doesn't exist!")

if (GEMINI_API_KEY):
    print(f"Gemini API key exists and starts with {GEMINI_API_KEY[:3]}")
else:
    print("Gemini API key doesn't exist!")

In [ ]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

#model="gpt-5-mini"
openai = OpenAI()
#model="gemini-3.8-flash"
gemini = OpenAI(base_url=gemini_url, api_key=GEMINI_API_KEY)
#model="deepseek-flash"
ollama = OpenAI(base_url=ollama_url, api_key="")

In [ ]:
agents = [
    {
        "name": "Alex",
        "client": openai,
        "model": "gpt-6-luna",
        "system_prompt": """
            You are Alex, an argumentative and skeptical participant.
            Challenge claims and point out weaknesses.
            You are talking with Blake and Charlie.
            Keep each response under 100 words.
            Return only what Alex says, without adding "Alex:".
            """,
        "extra_args": {
            "reasoning_effort": "medium"
        },
    },
     {
        "name": "Blake",
        "client": ollama,
        "model": "deepseek-r1:1.5b",
        "system_prompt": """
            You are Blake, a polite and optimistic participant.
            Look for common ground while giving clear reasons.
            You are talking with Alex and Charlie.
            Keep each response under 100 words.
            Return only what Blake says, without adding "Blake:".
            """,
        "extra_args": {
            "reasoning_effort": "medium"
        },
    },
    {
        "name": "Charlie",
        "client": gemini,
        "model": "gemini-3.5-flash-lite",
        "system_prompt": """
            You are Charlie, a neutral and analytical participant.
            Evaluate Alex's and Blake's arguments and add useful evidence.
            Keep each response under 100 words.
            Return only what Charlie says, without adding "Charlie:".
            """,
        "extra_args": {
            "reasoning_effort": "medium"
        },
    },
]

In [ ]:
conversation = [
    {
        "speaker": "Moderator",
        "content": (
            "Discuss this question: Should universities allow students "
            "to use generative AI for assignments?"
        ),
    }
]

In [ ]:
def format_conversation(conversation):
    lines = []

    for turn in conversation:
        speaker = turn["speaker"]
        content = turn["content"]
        lines.append(f"{speaker}: {content}")
    
    return "\n\n".join(lines) 

In [ ]:
def build_messages(agent, conversation):
    transcript = format_conversation(conversation)

    user_prompt = f"""
    You are {agent["name"]}, participating in a conversation with the other speakers.

    Here is the complete conversation so far:

    --- CONVERSATION START ---
    {transcript}
    --- CONVERSATION END ---

    Respond to the conversation with what {agent["name"]} should say next.
    Do not repeat the transcript.
    Do not speak for the other participants.
    Return only your next contribution.
    """

    return [
        {
            "role": 'system',
            "content": agent["system_prompt"]
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

In [ ]:
def call_agent(agent, conversation):
    messages = build_messages(agent, conversation)

    response = agent["client"].chat.completions.create(
        model=agent["model"],
        messages=messages,
        **agent["extra_args"],
    )

    reply = response.choices[0].message.content
    return reply.strip()

In [ ]:
NUMBER_OF_ROUNDS = 2

conversation = [
    {
        "speaker": "Moderator",
        "content": (
            "Discuss this question: Should universities allow students "
            "to use generative AI for assignments?"
        ),
    }
]

for round in range(NUMBER_OF_ROUNDS):
    for agent in agents:
        reply = call_agent(agent, conversation)

        conversation.append({
            "speaker": agent["name"],
            "content": reply
        })

In [ ]:
print(format_conversation(conversation))